# 03 - Bronze, Silver, Gold

Objetivo: construir um pipeline lakehouse pequeno, parecido com o que vais ver em Databricks.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

java_home = Path("/usr/local/opt/openjdk@17")
if not java_home.exists():
    java_home = Path("/opt/homebrew/opt/openjdk@17")

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = f"{java_home / 'bin'}:{os.environ['PATH']}"
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

print(f"Python: {sys.executable}")
print(f"JAVA_HOME: {os.environ['JAVA_HOME']}")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round as spark_round, sum as spark_sum, to_date

In [ ]:
spark = (
    SparkSession.builder.appName("notebook-03-bronze-silver-gold")
    .master("local[*]")
    .getOrCreate()
)


## Bronze

Bronze guarda dados quase crus. Aqui lemos CSV e escrevemos Parquet.

In [ ]:
def read_raw_csv(name):
    return (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .csv(str(PROJECT_ROOT / f"data/raw/{name}.csv"))
    )


bronze_orders = read_raw_csv("orders")
bronze_customers = read_raw_csv("customers")
bronze_products = read_raw_csv("products")

bronze_orders.write.mode("overwrite").parquet(str(LAKEHOUSE / "bronze/orders"))
bronze_customers.write.mode("overwrite").parquet(str(LAKEHOUSE / "bronze/customers"))
bronze_products.write.mode("overwrite").parquet(str(LAKEHOUSE / "bronze/products"))

## Silver

Silver limpa tipos, filtra dados invalidos e prepara as tabelas para consumo.

In [ ]:
silver_orders = (
    bronze_orders.withColumn("order_date", to_date(col("order_date")))
    .withColumn("quantity", col("quantity").cast("int"))
    .filter(col("quantity") > 0)
)

silver_orders.write.mode("overwrite").parquet(str(LAKEHOUSE / "silver/orders"))
silver_orders.show(truncate=False)

## Gold

Gold contem metricas prontas para reporting ou dashboards.

In [ ]:
gold_sales_by_customer = (
    silver_orders.filter(col("status") == "delivered")
    .join(bronze_customers, on="customer_id", how="left")
    .join(bronze_products, on="product_id", how="left")
    .withColumn("revenue", spark_round(col("quantity") * col("unit_price"), 2))
    .groupBy("customer_id", "name", "country", "segment")
    .agg(
        spark_sum("quantity").alias("items_sold"),
        spark_round(spark_sum("revenue"), 2).alias("total_revenue"),
    )
    .orderBy(col("total_revenue").desc())
)

gold_sales_by_customer.write.mode("overwrite").parquet(
    str(LAKEHOUSE / "gold/sales_by_customer")
)

gold_sales_by_customer.show(truncate=False)

## Em Databricks

No Databricks, trocar Parquet por Delta seria natural:

```python
df.write.mode("overwrite").format("delta").saveAsTable("catalog.schema.table")
```